In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "telecom_support_knowledge_base.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print(f"Loaded {len(pages)} pages from the PDF.")
print("\n-- First page preview (first 500 chars) ---")
print(pages[0].page_content[:500])

C:\Users\DELL\AppData\Local\Temp\ipykernel_2780\4208679438.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 9 pages from the PDF.

-- First page preview (first 500 chars) ---
Telecom Customer Support Knowledge Base - Mock document for RAG practice
Page 1
 Telecom Customer Support
 and FAQ Knowledge Base
 Version 3.2 | Internal reference for support agents and the AI assistant
About this document. This is a synthetic knowledge base created for learning and testing
Retrieval-Augmented Generation (RAG) pipelines. All numbers, plan names and policies are fictional. It is
not the original PDF from any course.
Contents
 - Section 1 - Billing and Account Management (page 2)


##### We create chunks -> for chunks we create embedding -> store in vectordb

#### overlap - each chunks has overlaps between them . chunk_size = 600 means Each generated chunk will aim to be a maximum of 600 characters. chunk_overlap=100 means Adjacent chunks will share 100 characters of overlap between them(means last 100 chars of chunk1 will also be stored in chunk2 as first 100 chars).separators = ["\n\n", "\n", ".", " "] it means This is the fallback priority order the splitter uses when trying to break text apart cleanly.

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 600, # ~ 150 words per chunk
    chunk_overlap = 100, # overlap keeps context at boundaries
    separators = ["\n\n","\n","."," "], # tries paragraph -> line -> sentence -> word
)
chunks = splitter.split_documents(pages)
len(chunks)

30

In [7]:
chunks[0].page_content

'Telecom Customer Support Knowledge Base - Mock document for RAG practice\nPage 1\n Telecom Customer Support\n and FAQ Knowledge Base\n Version 3.2 | Internal reference for support agents and the AI assistant\nAbout this document. This is a synthetic knowledge base created for learning and testing\nRetrieval-Augmented Generation (RAG) pipelines. All numbers, plan names and policies are fictional. It is\nnot the original PDF from any course.\nContents\n - Section 1 - Billing and Account Management (page 2)\n- Section 2 - Plans, Recharges and Validity (page 3)'

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# fetching the model (text is passed to this object, it converts sentences into dense mathematical vectors)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
# we are generating embeddings(means coordinates for every chunk) and store in vector DB
vector_store = Chroma.from_documents(chunks,embeddings)

print(f"Vector store ready. {vector_store._collection.count()} vectors stored.")

D:\tutorial-agentic-ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
D:\tutorial-agentic-ai\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate de

Vector store ready. 30 vectors stored.


In [9]:
retriever = vector_store.as_retriever(search_kwargs = {"k":3})

test_query = "What is VoLTE and how does it improve call quality?"
retrieved = retriever.invoke(test_query)

In [11]:
for i,doc in enumerate(retrieved,1):
    print(f"--- Chunk {i} ----")
    print(doc.page_content[:300])
    print()

--- Chunk 1 ----
Visit service center
Calls drop
Enable VoLTE / VoWiFi
Report tower issue
No OTP
Send START 0 to 1909
Raise SMS ticket
Data slow
Check quota, APN
Raise data ticket

--- Chunk 2 ----
Telecom Customer Support Knowledge Base - Mock document for RAG practice
Page 3
Section 2: Plans, Recharges and Validity
2.1 What prepaid plan categories are available?
 - Unlimited plans: daily data quota, unlimited calls and 100 SMS per day, with validity of 28, 56, 84 or
365 days.
- Data booster 

--- Chunk 3 ----
Telecom Customer Support Knowledge Base - Mock document for RAG practice
Page 9
Section 8: Complaints, Escalation and Service Levels
8.1 How do I raise a complaint?
 - Use the app (Help > Raise Ticket), call 198, or visit an Experience Center. Every complaint receives a
ticket number by SMS.
- Keep 



In [12]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
#from langchain_groq import ChatGroq
from langchain_google_genai import ChatGoogleGenerativeAI

def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

SYSTEM_PROMPT = """\
You are a helpful telecom assistant.
Answer the question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.

Context
{context}
"""
prompt = ChatPromptTemplate.from_messages([

    ("system",SYSTEM_PROMPT),
    ("human","{question}"),
])

llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash", 
    temperature=0,
)

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain assembled with Gemini.")

RAG chain assembled with Gemini.


In [13]:
question = "How does international roaming work and what charges should I expect?"
print(f"Q : {question}\n")
print("A:",chain.invoke(question))


Q : How does international roaming work and what charges should I expect?

A: Based on the provided context, here is how international roaming works and the charges you can expect:

### How International Roaming Works
* **Activation:** Activate international roaming from the app under **Services > International Roaming** at least 24 hours before traveling. 
  * Prepaid users must have a minimum balance of **Rs. 500**.
  * Postpaid users must have no overdue bills.
* **Usage Setup:** Once you land, restart your phone and enable **Data Roaming** in your phone settings. If automatic network selection fails, manually select a partner network.

---

### Charges to Expect

#### 1. International Roaming Packs
You can purchase one of the following standard roaming packs:
* **Roam 1:** **Rs. 799** (Valid for 1 day | Includes 1 GB data, 100 minutes, 100 SMS)
* **Roam 7:** **Rs. 2,999** (Valid for 7 days | Includes 5 GB data, 300 minutes, 100 SMS)
* **Roam 30:** **Rs. 5,999** (Valid for 30 days |